# Task 2: Original LCS System on raw Dataset

## Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score
from skeLCS import eLCS
import time

## Loading the raw dataset

In [ ]:
#Load the raw dataset for the LCS baseline
df_raw_lcs = pd.read_csv('Yaacoub_creditcard.csv')
df_raw_lcs.head()

## Minimal processing + stratified subsample

In [ ]:
#Minimal processing needed for LCS compatibility
#separate features/target, take a stratified subsample for computational feasibility
X = df_raw_lcs.drop(columns=['Class']).values
y = df_raw_lcs['Class'].values

_, X_sample, _, y_sample = train_test_split(X, y, test_size=15000, stratify=y, random_state=42)

X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, test_size=0.2, stratify=y_sample, random_state=42)

print(f"Training set: {X_train.shape}, Fraud cases: {y_train.sum()}")
print(f"Test set: {X_test.shape}, Fraud cases: {y_test.sum()}")

## Run the original, unmodified eLCS baseline

In [ ]:
model = eLCS(learning_iterations=10000, random_state=42)

start = time.time()
model.fit(X_train, y_train)
elapsed = time.time() - start

preds = model.predict(X_test)

print(f"Training time: {elapsed:.2f} seconds")
print(f"Accuracy: {accuracy_score(y_test, preds):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, preds):.4f}")
print(f"Precision: {precision_score(y_test, preds, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, preds, zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_test, preds, zero_division=0):.4f}")

### Task 2: Original LCS System on Raw Dataset (Results)
An original, unmodified eLCS system (scikit-eLCS implementation) was run on the raw dataset prior to any cleaning or transformation. Due to the computational cost of LCS rule evolution on the full 284,807-row dataset, a stratified subsample of 15,000 rows was used, preserving the original ~0.17% fraud rate (21 fraud cases in the 12,000-row training set; 5 in the 3,000-row test set). This subsampling, along with separating features and the target label into NumPy arrays, was the only processing applied. No cleaning, scaling, or transformation was done, in line with the requirement to establish a baseline on raw data. 

The model was trained using eLCS's default parameters (learning_iterations = 10,000, N = 1,000, p_spec = 0.5, nu = 5, chi = 0.8, mu = 0.04, theta_GA = 25) with no modification. Training took 39.41 seconds.  

**Baseline results:**  
|  Metric  |  Value  |  
|----------|---------|  
| Accuracy | 0.9983  |  
| Balanced Accuracy | 0.5000 |  
|Precision | 0.0000  |  
| Recall   | 0.0000  |  
| F1-score | 0.0000  |  

Although accuracy is very high, it's misleading given the class imbalance. A classifier that predicts legitimate for every transaction would have similar accuracy. The balanced accuracy of 0.5, alongside zero precision, recall, and F1-score, shows the original LCS system failed to identify any fraudulent transactions in the test set, consistent with the small number of fraud examples in the raw, unprocessed training data (21 cases). This suggests the system didn't encounter enough fraud examples to evolve rules capable of distinguishing fraud from legitimate transactions. This result supports the preprocessing and improvement work done in Tasks 3 and 4.

# Task 3 Data Preprocessing and Feature Engineering

## Leakage-Safe Train/Test Split 

Missing values, invalid values, inconsistent categories, and data types were addressed in Phase 1, where none were found. The 1,081 duplicate rows identified in Phase 1 are removed here before splitting, to prevent identical transactions appearing in both training and test sets. The dataset is split into training and test sets before any scaling or resampling is applied, ensuring no information from the test set influences preprocessing decisions.  

Outliers in Amount, investigated using the IQR method in Phase 1, was retained rather than removed, as they represented genuine high-value transactions rather than invalid data, and fraud was only marginally overrepresented among them.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

#Start from the raw dataset again (not Phase 1's df_final to keep the pipeline self-contained and leak-free)
df_lcs = df_raw_lcs.drop_duplicates().copy()

X = df_lcs.drop(columns=['Class'])
y = df_lcs['Class']

#Split FIRST, before any scaling/transformation, to avoid test-set leakage
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Train shape: {X_train_raw.shape}, Fraud cases: {y_train.sum()}")
print(f"Test shape: {X_test_raw.shape}, Fraud cases: {y_test.sum()}")

## Scaling and Transformation  

Amount is log-transformed to reduce its severe right-skew. Time and the log-transformed Amount are standardised using StandardScaler, fitted only on the training set and then applied to the test set using the training-derived parameters. This avoids leaks, since the test sets transformation doesn't depend on its own statistics.

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

#Log-transform Amount on both sets (log1p is a fixed mathematical function, not fitted to the data, so no leakage)
X_train_raw = X_train_raw.copy()
X_test_raw = X_test_raw.copy()

X_train_raw['Amount_log'] = np.log1p(X_train_raw['Amount'])
X_test_raw['Amount_log'] = np.log1p(X_test_raw['Amount'])

#Fit scaler ONLY on training data, then apply to both
scaler_time = StandardScaler()
scaler_amount = StandardScaler()

X_train_raw['Time_scaled'] = scaler_time.fit_transform(X_train_raw[['Time']])
X_test_raw['Time_scaled'] = scaler_time.transform(X_test_raw[['Time']]) #transform only, not fit

X_train_raw['Amount_scaled'] = scaler_amount.fit_transform(X_train_raw[['Amount_log']])
X_test_raw['Amount_scaled'] = scaler_amount.transform(X_test_raw[['Amount_log']]) #transform only, not fit

#Final feature set
final_cols = ['Time_scaled'] + [f'V{i}' for i in range(1, 29)] + ['Amount_scaled']
X_train_final = X_train_raw[final_cols]
X_test_final = X_test_raw[final_cols]

print(f"X_train_final shape: {X_train_final.shape}")
print(f"X_test_final shape: {X_test_final.shape}")


## Addressing Class Imbalance  

Class imbalance is addressed using SMOTE, applied only to the training set after the split. This creates synthetic fraud examples by interpolating between existing fraud cases. The test set stays untouched, keeping the original fraud rate, so evaluation reflects real-world class distribution rather than the artificially balanced training data.

In [ ]:
from imblearn.over_sampling import SMOTE

print(f"Before SMOTE - Training set: {X_train_final.shape}, Fraud cases: {y_train.sum()}")

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_final, y_train)

print(f"After SMOTE - Training set: {X_train_resampled.shape}, Fraud cases: {y_train_resampled.sum()}")
print(f"Legitimate cases: {(y_train_resampled == 0).sum()}")

## Feature Selection  

A reduced feature set is prepared as an additional preprocessing variant, directed by Phase 1's correlation analysis, which noted V17, V14, V12, V10, V11, and V4 as the features strongly correlated with Class. This 8 feature set (including Time_scaled and Amount_scaled) is tested alongside the full 30-feature set in Task 4, to test whether reduction improves LCS performance.

In [ ]:
#Feature selection based on Phase 1 correlation analysis:
# V17, V14, V12, V10, V11, V4 showed the strongest correlation with Class
top_features = ['V17', 'V14', 'V12', 'V10', 'V11', 'V4', 'Time_scaled', 'Amount_scaled']

X_train_selected = X_train_resampled[top_features]
X_test_selected = X_test_final[top_features]

print(f"X_train_selected shape: {X_train_selected.shape}")
print(f"X_test_selected shape: {X_test_selected.shape}")

### Task 3: Data Preprocessing and Feature Engineering (Results)  

This preprocessing is expected to improve LCS performance in many ways. eLCS's rule-discovery process depends on seeing enough examples of the minority class to evolve rules that cover it. Task 2's baseline showed the model failed to detect any fraud when trained on the raw, imbalanced data. SMOTE addresses this by providing a much larger and balanced set of fraud examples to learn from. Scaling Time and Amount to match the standardised range of the PCA components should also help these features contribute more evenly to rule matching, rather than being overshadowed by features on different numeric scales. Finally, the reduced feature set is expected to shrink the LCS search space, potentially allowing rules to specialise more effectively around the small number of features known to carry the strongest fraud signal.

# Task 4: Improved LCS-Based System

In [ ]:
from sklearn.utils import resample

#Subsample from the SMOTE-balanced training set for computational feasibility,
#keeping the 50/50 balance intact (Different from Task 2's subsample which kept the raw ~0.17% imbalance)
X_train_bal_sub, y_train_bal_sub = resample(X_train_resampled, y_train_resampled, n_samples=15000, stratify=y_train_resampled, random_state=42)

print(f"Balanced training subsample: {X_train_bal_sub.shape}, Fraud cases: {y_train_bal_sub.sum()}")

In [ ]:
from skeLCS import eLCS
import time
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score

#Improved system: same eLCS algorithm, but trained on preprocessed + balanced data,
#with increased learning_iterations (hyperparameter tuning)
improved_model = eLCS(learning_iterations=20000, random_state=42)

start = time.time()
improved_model.fit(X_train_bal_sub.values, y_train_bal_sub.values)
elapsed = time.time() - start

#Evaluate on the UNTOUCHED, real-world-imbalanced test set
preds_improved = improved_model.predict(X_test_final.values)

print(f"Training time: {elapsed:.2f} seconds")
print(f"Accuracy: {accuracy_score(y_test, preds_improved):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, preds_improved):.4f}")
print(f"Precision: {precision_score(y_test, preds_improved, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, preds_improved, zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_test, preds_improved, zero_division=0):.4f}")

In [ ]:
top_features = ['V17', 'V14', 'V12', 'V10', 'V11', 'V4', 'Time_scaled', 'Amount_scaled']

#Use the SAME subsampled rows as the full-feature model, just the reduced columns
#This keeps the comparison fair. (Identical training examples, only feature count differs)
X_train_bal_sub_selected = X_train_bal_sub[top_features]
X_test_selected_final = X_test_final[top_features]

print(f"Reduced-feature training subsample: {X_train_bal_sub_selected.shape}")
print(f"Reduced-feature test set: {X_test_selected_final.shape}")

In [ ]:
improved_model_selected = eLCS(learning_iterations=20000, random_state=42)

start = time.time()
improved_model_selected.fit(X_train_bal_sub_selected.values, y_train_bal_sub.values)
elapsed = time.time() - start

preds_selected = improved_model_selected.predict(X_test_selected_final.values)

print(f"Training time: {elapsed:.2f} seconds")
print(f"Accuracy: {accuracy_score(y_test, preds_selected):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, preds_selected):.4f}")
print(f"Precision: {precision_score(y_test, preds_selected, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, preds_selected, zero_division=0):.4f}")
print(f"F1 Score: {f1_score(y_test, preds_selected, zero_division=0):.4f}")

### Task 4: Improved LCS-Based System(Results)  

The improved system combines three changes relative to the original baseline (Task 2): class-imbalance handling via SMOTE (applied only to the training set), preprocessing designed for LCS (scaling Time and Amount to match the standardised PCA components), and increased hyperparameter tuning (learning_iterations raised from 10,000 to 20,000). A second variant was also tested, additionally applying feature selection to reduce the feature set from 30 to 8 (the six PCA components most correlated with Class from Phase 1 plus Time_scaled and Amount_scaled).  

Why these changes should improve performance: the baseline's complete failure to detect fraud (recall = 0) was tied to eLCS's rule-discovery process having too few minority-class examples to evolve effective rules. SMOTE addresses this by balancing the training data. Scaling Time and Amount ensures these features are not overshadowed by the standardised PCA components during rule matching. Increased learning iterations allow the rule population more chances to converge given the larger, balanced training set. Feature selection, in the second variant, reduces the search space, possibly allowing rules to specialise more efficiently around the features already known to carry the strongest signal.  

Both improved variants were evaluated on the same untouched, real-world-imbalanced test set used in Task 2, ensuring a fair comparison.  

| Metric | Baseline | Improved (30 features) | Improved (8 features) |
|--------|----------|------------------------|-----------------------|
| Training time | 39.41s | 76.97s | 42.27s |
| Accuracy | 0.9983 | 0.9837 | 0.9879 |
| Balanced Accuracy | 0.5000 | 0.9183 | 0.9099 |
| Precision | 0.0000 | 0.0817 | 0.1053 |
| Recall | 0.0000 | 0.8526 | 0.8316 |
| F1-score | 0.0000 | 0.1490 | 0.1870 |  

Both improved variants display an improvement over the baseline: balanced accuracy rose from chance level (0.50) to over 0.90, and recall rose from 0.00 to over 0.83, meaning the improved system now properly identifies the majority of fraudulent transactions in the test set, addressing the false-negative cost concern raised in Phase 1's motivation. This comes at the cost of reduced precision, a known and expected trade-off of SMOTE-based rebalancing, since the model becomes more willing to flag transactions as fraudulent.

Between the two improved variants, the 8-feature model trained approximately 42% faster while achieving a higher F1-score (0.187 vs 0.149) and better precision, at the cost of a small reduction in recall (0.832 vs 0.853) and balanced accuracy. This suggests that the feature selection informed by Phase 1's correlation analysis provides a good trade-off between predictive performance and computational cost, supporting its use as the preferred configuration for the improved system moving forward.

# Task 5: Experiments  



## Metrics justification

In [ ]:
#Regenrate baseline predictions on the SAME test set used for the improved model,
# using the raw (unscaled) columns in their original order to match the baseline model's training format
raw_cols_order = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount']
X_test_raw_for_baseline = X_test_raw[raw_cols_order]

preds_baseline_full = model.predict(X_test_raw_for_baseline.values)
baseline_probs = model.predict_proba(X_test_raw_for_baseline.values)[:, 1]

print(f"Baseline predictions generated on full test set: {len(preds_baseline_full)}")

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score

#Baseline_probs already generated above
improved_probs = improved_model.predict_proba(X_test_final.values)[:, 1]

print("Baseline Confusion Matrix:")
print(confusion_matrix(y_test, preds_baseline_full))
print(f"Baseline ROC AUC: {roc_auc_score(y_test, baseline_probs):.4f}")
print(f"Baseline PR-AUC: {average_precision_score(y_test, baseline_probs):.4f}")

print("\nImproved (30-feature) Confusion Matrix:")
print(confusion_matrix(y_test, preds_improved))
print(f"Improved ROC AUC: {roc_auc_score(y_test, improved_probs):.4f}")
print(f"Improved PR-AUC: {average_precision_score(y_test, improved_probs):.4f}")

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar
import numpy as np

#Build the 2x2 contingency table of correct/incorrect predictions between the two models
baseline_correct = (preds_baseline_full == y_test)
improved_correct = (preds_improved == y_test)

both_correct = np.sum(baseline_correct & improved_correct)
baseline_only = np.sum(baseline_correct & ~improved_correct)
improved_only = np.sum(~baseline_correct & improved_correct)
both_wrong = np.sum(~baseline_correct & ~improved_correct)

contingency_table = np.array([[both_correct, baseline_only],[improved_only, both_wrong]])

print("Contingency Table:")
print(contingency_table)

result = mcnemar(contingency_table, exact=True)
print(f"\nMcNemar's test statistic: {result.statistic}")
print(f"p-value: {result.pvalue}")

### Task 5: Experiments(Results)

Validation Strategy: An 80/20 stratified train-test split was used, rather than k-fold cross-validation mainly due to the computational cost of LCS training and prediction(single training runs took 40-75 seconds even on subsampled data (Tasks 2 and 4)), and generating predictions for the full 56,746-row test set took approximately 25 minutes per model. K-fold cross-validation would multiply this cost by the number of folds, making it impractical within the project's time constraints. Stratification confirmed the severe class imbalance (0.167% fraud) was preserved proportionally in both the training and test sets.

Evaluation Metrics: Given the class imbalance seen in Phase 1, accuracy alone was deemed unsuitable, as it can appear high even when a model fails to detect any fraud (seen by the baseline's 99.83% accuracy despite zero recall). Balanced accuracy, precision, recall, and F1-score were prioritised, alongside ROC-AUC and PR-AUC, with emphasis on PR-AUC as the more informative metric on imbalanced data. Confusion matrices were also looked at, since they reveal the practical cost of each error type (missed fraud vs. false alarms) discussed as a key trade-off in Phase 1's motivation.

Results:

| Metric | Baseline | Improved (30 features) |
|--------|----------|------------------------|
| Confusion Matrix | [[56648, 3], [95, 0]] | [[55740, 911], [14, 81]] |
| ROC-AUC | 0.6578 | 0.9656 |
| PR-AUC | 0.2223 | 0.5834 |

The baseline model's confusion matrix confirms it failed to detect any of the 95 fraudulent transactions in the test set (0 true positives). The improved model correctly identified 81 of these 95 cases, at the cost of 911 false positives among legitimate transactions. ROC-AUC improved from 0.658 (barely above the 0.5 chance level) to 0.966, and PR-AUC, the more appropriate metric given the class imbalance, improved substantially from 0.222 to 0.583.

Statistical Test: McNemar's test was used to compare the paired predictions of the baseline and improved models on the identical test set, since it is suited to comparing two classifiers' correct/incorrect outcomes on matched samples. The test produced a statistic of 81.0 and a p-value of 1.02 × 10⁻¹⁷⁷, far below the α = 0.05 significance threshold. This provides strong statistical evidence that the improved system's performance differs majorly from the baseline, confirming that the observed gains are not attributable to chance.

# Task 6: Model Comparison


In [ ]:
import time

print(f"Starting fit at {time.strftime('%H:%M:%S')}", flush=True)
model_preprocessed = eLCS(learning_iterations=10000, random_state=42)

start = time.time()
model_preprocessed.fit(X_train_bal_sub.values, y_train_bal_sub.values)
elapsed = time.time() - start
print(f"Fit finished at {time.strftime('%H:%M:%S')} (took {elapsed:.2f}s)", flush=True)

print(f"Starting predict at {time.strftime('%H:%M:%S')}", flush=True)
predict_start = time.time()
preds_preprocessed = model_preprocessed.predict(X_test_final.values)
predict_elapsed = time.time() - predict_start
print(f"predict finished at {time.strftime('%H:%M:%S')} (took {predict_elapsed:.2f}s)", flush=True)

print(f"Starting predict_proba at {time.strftime('%H:%M:%S')}", flush=True)
proba_start = time.time()
probs_preprocessed = model_preprocessed.predict_proba(X_test_final.values)[:, 1]
proba_elapsed = time.time() - proba_start
print(f"predict_proba finished at {time.strftime('%H:%M:%S')} (took {proba_elapsed:.2f}s)", flush=True)

print(f"\nTraining time: {elapsed:.2f} seconds")
print(f"Accuracy: {accuracy_score(y_test, preds_preprocessed):.4f}")
print(f"Balanced Accuracy: {balanced_accuracy_score(y_test, preds_preprocessed):.4f}")
print(f"Precision: {precision_score(y_test, preds_preprocessed, zero_division=0):.4f}")
print(f"Recall: {recall_score(y_test, preds_preprocessed, zero_division=0):.4f}")
print(f"F1-score: {f1_score(y_test, preds_preprocessed, zero_division=0):.4f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

conventional_models = {
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42)
}

conventional_results = {}

for name, clf in conventional_models.items():
    start = time.time()
    clf.fit(X_train_resampled, y_train_resampled)  # full SMOTE-balanced training set (not subsampled(these models scale fine))
    elapsed = time.time() - start
    
    preds = clf.predict(X_test_final)
    probs = clf.predict_proba(X_test_final)[:, 1]
    
    conventional_results[name] = {
        'time': elapsed,
        'accuracy': accuracy_score(y_test, preds),
        'balanced_accuracy': balanced_accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall': recall_score(y_test, preds, zero_division=0),
        'f1': f1_score(y_test, preds, zero_division=0),
        'roc_auc': roc_auc_score(y_test, probs),
        'pr_auc': average_precision_score(y_test, probs)
    }
    print(f"{name}: {conventional_results[name]}")

In [ ]:
print(f"Original LCS (preprocessed) ROC-AUC: {roc_auc_score(y_test, probs_preprocessed):.4f}")
print(f"Original LCS (preprocessed) PR-AUC: {average_precision_score(y_test, probs_preprocessed):.4f}")

| Model | Training Time | Accuracy | Balanced Accuracy | Precision | Recall | F1 | ROC-AUC | PR-AUC |
|-------|---------------|----------|-------------------|-----------|--------|----|---------|--------|
Original LCS (raw) | 39.41s | 0.9983 | 0.5000 | 0.0000 | 0.0000 | 0.0000 | 0.6578 |	0.2223 |
Original LCS (preprocessed) | 35.05s | 0.9884|	0.9154|	0.1110|	0.8421|	0.1961|	0.9668|	0.5506|
Improved LCS (30 features)|	76.97s|	0.9837	|0.9183|	0.0817|	0.8526|	0.1490|	0.9656|	0.5834|
Random Forest|	409.02s|	0.9995|	0.8736|	0.9221|	0.7474|	0.8256|	0.9691|	0.8122|
Logistic Regression|	1.12s|	0.9735|	0.9237|	0.0528|	0.8737|	0.0996| 0.9592|	0.6829|
Decision Tree|	49.62s	|0.9976|	0.8464|	0.3771|	0.6947|	0.4889|	0.8464|	0.2625|

### Task 6: Model Comparison(Results)

The proposed improved LCS-based system was compared against the original LCS on both raw and preprocessed data, and against three conventional non-deep-learning models (Random Forest, Logistic Regression, and Decision Tree), all trained on the same SMOTE-balanced training data and evaluated on the untouched, real-world-imbalanced test set.

The original LCS on raw data (Task 2) failed completely, with balanced accuracy at chance level (0.500) and zero recall. Applying preprocessing alone (SMOTE, scaling) to the original, unmodified LCS system produced an improvement, with balanced accuracy rising to 0.9154 and recall to 0.8421 which is nearly matching the fully improved system's performance (balanced accuracy 0.9183, recall 0.8526). This indicates that preprocessing, rather than the additional hyperparameter tuning applied in Task 4, was the primary driver of performance improvement; the gain from tuning alone was small by comparison.

Among the conventional models, Random Forest was the standout performer, achieving the highest accuracy (0.9995), by far the best precision (0.9221), and the best ROC-AUC (0.9691) and PR-AUC (0.8122) of all six models tested, though at a substantially higher training cost (409.02 seconds) than the other conventional models. Logistic Regression achieved the highest recall of any model tested (0.8737) but the lowest precision (0.0528), reflecting its simple linear decision boundary combined with SMOTE-balanced training data, which biases it toward predicting fraud. Decision Tree performed moderately across all metrics, without excelling on any single one.

Overall, the LCS-based models (preprocessed and improved) achieved a similar profile to Logistic Regression — high recall, low precision — while Random Forest achieved a better balance between the two, and the best overall discriminative ability (PR-AUC). This suggests that, for this dataset, conventional methods such as Random Forest may offer superior raw predictive performance compared to LCS-based approaches, though LCS retains an advantage in interpretability through its evolved rule sets, which is examined further in Task 7.

# Task 7: Interpretation of Results

In [ ]:
# Export the rules from the improved LCS model
improved_model.export_final_rule_population(X_test_final.columns, 'Class', 'improved_lcs_rules.csv', DCAL=True)

import pandas as pd
rules_df = pd.read_csv('improved_lcs_rules.csv')
print(f"Total rules in population: {len(rules_df)}")
rules_df.head(10)

In [ ]:
# Filter for fraud-predicting rules with genuine evidence of usefulness (Correct Count > 0)
fraud_rules = rules_df[(rules_df['Class'] == 1) & (rules_df['Correct Count'] > 0)]
fraud_rules_sorted = fraud_rules.sort_values(by=['Correct Count', 'Fitness'], ascending=False)
print(f"Fraud-predicting rules with match evidence: {len(fraud_rules_sorted)}")
fraud_rules_sorted.head(5)[['Specified Attribute Names', 'Class', 'Fitness', 'Accuracy', 'Correct Count', 'Match Count', 'Specificity']]

In [ ]:
# Pull the full detail (including specified value ranges) for the top rules
top_rules_detail = rules_df.loc[[16, 43, 78, 38, 57]][
    ['Specified Values', 'Specified Attribute Names', 'Class', 'Fitness', 'Accuracy', 'Correct Count', 'Match Count']
]
for idx, row in top_rules_detail.iterrows():
    print(f"\n--- Rule {idx} ---")
    print(f"Attributes: {row['Specified Attribute Names']}")
    print(f"Values: {row['Specified Values']}")
    print(f"Class: {row['Class']}, Fitness: {row['Fitness']:.4f}, Accuracy: {row['Accuracy']:.4f}")
    print(f"Correct: {row['Correct Count']}/{row['Match Count']}")

### Task 7: Interpretation of Results (Yaacoub)

**Interpreting the Model Comparison:** Task 6 showed that preprocessing was the primary force of improvement for LCS-based models, with the original LCS on preprocessed data almost matching the fully improved system's balanced accuracy and recall. Compared with conventional models, Random Forest achieved the best overall precision-recall balance and discriminative ability (PR-AUC = 0.8122), whilst LCS-based models shared a similar profile to Logistic Regression: high recall, low precision. This insinuates Random Forest may offer better raw predictive performance on this dataset, but LCS offers predictions that can be traced back to a specific, human-readable rule.

**Example LCS Rules:** Three rules were taken from the improved LCS system's final rule population (923 total rules), selected for high match counts and high accuracy:

Rule 1 (matched 4,443 times, 99.86% accuracy): predicts fraud when V2 is between -10.72 and 20.60, V6 between -14.78 and 5.20, V14 between -19.58 and -2.78, V18 between -9.67 and 0.28, V19 between -2.07 and 5.33, V23 between -10.18 and 7.53, V26 between -1.01 and 0.85, and V27 between -1.17 and 5.32.

Rule 2 (matched 4,513 times, 92.22% accuracy): predicts fraud when V12 is between -17.62 and -1.50, V16 between -22.60 and 3.60, V19 between -2.07 and 5.05, V21 between -16.38 and 17.17, V23 between -10.52 and 11.53, V24 between -2.10 and 2.41, and V28 between -5.46 and 5.38.

Rule 3 (matched 3,041 times, 99.31% accuracy): predicts fraud when V2 is between -24.07 and 19.17, V13 between -3.06 and 2.15, V14 between -20.18 and -3.24, V20 between -6.01 and 9.83, V25 between -0.99 and 1.92, and V26 between -1.42 and 2.16.

**Discussion of Decision-Making Process:** Each rule represents a combination of conditions across several features. A transaction is classified as fraudulent only if all its feature values fall within every specified range. V14 appears as a narrow, negative range in both Rules 1 and 3 (approximately -20 to -3), consistent with Phase 1's correlation analysis, where V14 (-0.293) was among the strongest negative correlation of fraud. V12 appears as a negative range in Rule 2, matching its Phase 1 correlation (-0.251). V19 and V23 recur across multiple top rules with narrow ranges despite showing negligible linear correlation with Class in Phase 1 (0.034 and -0.006 respectively), meaning LCS may be capturing non-linear or interaction-based patterns that a simple correlation coefficient would not detect.

**Supporting Interpretability:** Unlike Random Forest or Logistic Regression, which make predictions through aggregated votes across many trees or one global weighted combination of all features respectively, each LCS rule is a readable "if-then" statement. A specific prediction can be explained by pointing to the exact rule (or rules) that matched the transaction and the exact feature ranges responsible which is a practical advantage in a fraud-detection context, where a flagged transaction may need to be explained to a customer, bank officer, or regulator.

**Trustworthiness and Practical Meaningfulness:** This advantage comes with an important issue. Because V1 through V28 are PCA-transformed components with no disclosed real-world meaning (Phase 1), a rule such as "V14 between -19.58 and -2.78" cannot be translated into a business-interpretable statement like "transactions from this merchant category" or "purchases above a certain amount at unusual hours." The rules are mathematically precise and traceable, but not semantically meaningful to a human stakeholder without access to the original, undisclosed features. This limits the value of LCS's interpretability advantage for this specific dataset — the explanation is sound, but the content of the explanation is not actionable in a real-world deployment sense.